# Metrics Module Test Notebook

Validates `cuic_quant.metrics` against backtester-compatible output.


In [3]:
from pathlib import Path
import pandas as pd

from cuic_quant.metrics import (
    calculate_all_metrics,
    calculate_max_drawdown,
    calculate_profit_factor,
    calculate_sharpe_ratio,
    calculate_win_rate,
)


In [4]:
REQUIRED_BACKTESTER_COLUMNS = [
    'timestamp', 'game', 'action', 'bet_size',
    'odds', 'outcome', 'pnl', 'cumulative_pnl', 'bankroll',
]

root = Path.cwd()
if not (root / 'data').exists() and (root.parent / 'data').exists():
    root = root.parent

candidate_paths = [
    root / 'data' / 'backtest_results.csv',
    root / 'data' / 'dummy_backtest_output.csv',
]

selected_path = next((p for p in candidate_paths if p.exists()), None)
if selected_path is None:
    raise FileNotFoundError(
        'No backtester output found. Expected one of: ' + ', '.join(str(p) for p in candidate_paths)
    )

results = pd.read_csv(selected_path)
print(f'Loaded {len(results)} rows from {selected_path}')
print('Columns:', results.columns.tolist())


Loaded 5 rows from c:\Users\bengr\Desktop\CUIC_Sem2_Project\data\dummy_backtest_output.csv
Columns: ['timestamp', 'game', 'action', 'bet_size', 'odds', 'outcome', 'pnl', 'cumulative_pnl', 'bankroll']


In [5]:
missing = [c for c in REQUIRED_BACKTESTER_COLUMNS if c not in results.columns]
assert not missing, f'Missing expected backtester columns: {missing}'

metrics = calculate_all_metrics(results)
metrics


{'total_trades': 5,
 'win_rate': 0.6,
 'total_pnl': 60.0,
 'sharpe_ratio': 1.6067872122085052,
 'max_drawdown': 1.5789473684210527,
 'profit_factor': 1.2608695652173914}

In [6]:
# Individual metric checks
print('Sharpe:', calculate_sharpe_ratio(results['pnl']))
print('Max drawdown:', calculate_max_drawdown(results['cumulative_pnl']))
print('Win rate:', calculate_win_rate(results['outcome']))
print('Profit factor:', calculate_profit_factor(results['pnl']))


Sharpe: 1.6067872122085052
Max drawdown: 1.5789473684210527
Win rate: 0.6
Profit factor: 1.2608695652173914


In [7]:
# Edge-case checks from brief
assert calculate_sharpe_ratio(pd.Series(dtype=float)) == 0.0
assert calculate_profit_factor(pd.Series([1.0, 2.0])) == float('inf')
assert calculate_profit_factor(pd.Series([-1.0, -2.0])) == 0.0
print('Edge-case checks passed')


Edge-case checks passed
